In [1]:
import pandas as pd
import numpy as np
import rasterio
import quantile_forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from quantile_forest import RandomForestQuantileRegressor

import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import time
import os

# Load the CSV file from the specified relative path
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial.csv')

# Check the columns in the DataFrame
print(lsms_spatial.columns)


# ------------------------------------------------------------------------------
# Two models for stacking/ensembling, starting with the overall quantile random forest with ranger

start_time = time.time()

# Prepare data for training
X = lsms_spatial.drop(columns=['farm_area_ha'])
y = lsms_spatial['farm_area_ha']

## Standardize the data
#scaler = StandardScaler()
#X_scaled = scaler.fit_transform(X)

# Train Random Forest model
rf = RandomForestRegressor(n_estimators=1500, max_features=3, min_samples_leaf=50, random_state=2024)
cv_scores = cross_val_score(rf, X, y, cv=10)
rf.fit(X, y)
print('---------------------RF training is complete----------------------------------')
## Quantile Random Forest (using RandomForestRegressor as a placeholder)
## Note: For actual quantile regression, consider using libraries like `quantile_forest` or `lightgbm` with quantile objectives
#qrf = RandomForestRegressor(n_estimators=1500, max_features=3, min_samples_leaf=50, random_state=2024)
#qrf.fit(X_scaled, y)


# Initialize the QuantileForestRegressor with the specified parameters
qrf = RandomForestQuantileRegressor(n_estimators=1500, max_features=3, min_samples_leaf=50, random_state=2024)

# Fit the model to the data
qrf.fit(X, y)
print('---------------------QRF training is complete----------------------------------')

end_time = time.time()
print(f"Training time: {end_time - start_time} seconds")

# Save models
import joblib
joblib.dump(rf, '../data/processed/rf_best_model.pkl')
joblib.dump(qrf, '../data/processed/qrf_best_model.pkl')

## -----------------------------------------------------------------------------
## Send email when script is completed
#recipients = 'd.hougni@cgiar.org'
#sender = 'test@noreply.com'
#
## Full email message
#message = MIMEMultipart()
#message['From'] = 'R (curl package) <test@noreply.com>'
#message['To'] = 'Deo <d.hougni@cgiar.org>'
#message['Subject'] = 'R script completed on CGLAB - RStudio terminal'
#
#body = '06.1.1.adhoc_basic_quantile_randomForest_model.R is completed!!!\n\nThe above-mentioned R script is completed\n\nThis email was sent using smtplib.'
#message.attach(MIMEText(body, 'plain'))
#
## Send the email
#try:
#    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
#        server.login('curlpackage', 'qyyjddvphjsrbnlm')
#        server.sendmail(sender, recipients, message.as_string())
#    print("Email sent successfully")
#except Exception as e:
#    print(f"Failed to send email: {e}")

Index(['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market'],
      dtype='object')
---------------------RF training is complete----------------------------------
---------------------QRF training is complete----------------------------------
Training time: 5340.15541768074 seconds


['../data/processed/qrf_best_model.pkl']